## Descarga de dependencias.

In [6]:
!pip install --no-cache-dir --upgrade unsloth unsloth_zoo torch datasets trl ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 126.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 240.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 376.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 179.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 86.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 296.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.3/217.3 kB 345.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 242.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 89.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 141.0 MB/s eta 0:00:00 0:00:01
   ━

## Prueba de modelo (Interferencia) 

In [11]:
# =============================================================================
# Qwen3 0.6B: entrenamiento + evaluación + prueba + exportación GGUF (Kaggle)
#
# Adaptado desde la versión Gemma 3:
#   - Se usa el chat template nativo de Qwen3 (ChatML: <|im_start|>/<|im_end|>).
#     Se deja el template por defecto (sin bloque <think>) tanto al entrenar como
#     al probar, así entrenamiento e inferencia son idénticos y el GGUF lleva
#     el mismo template.
#   - Máscara de pérdida: instruction_part / response_part de ChatML.
#   - Token de fin de turno: <|im_end|> (Qwen no usa <bos>).
#   - Qwen3 es estable en fp16, se quita la advertencia específica de Gemma.
#   - Las entradas se normalizan (strip + minúsculas) al entrenar, probar y
#     evaluar. La app Android debe hacer lo mismo.
#   - Se omiten ejemplos con la clave "NOTA".
# =============================================================================

from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

import gc
import json
import zipfile
from pathlib import Path

import torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
import ipywidgets as widgets
from IPython.display import display, FileLink, HTML

RUTA_BASE = Path("/kaggle/input/")

# ----------------------------- Configuración ---------------------------------
MODELO_BASE = "unsloth/Qwen3-0.6B"
NOMBRE_FAMILIA = "Qwen3"
MAX_SEQ_LENGTH = 2048
EPOCAS = 12
LEARNING_RATE = 2e-4
LORA_R = 32
LORA_ALPHA = 32
BATCH_SIZE = 2
ACUMULACION = 1
# "q8_0" es más fiel; "q4_k_m" pesa menos pero degrada más en modelos pequeños.
CUANTIZACION_GGUF = "q8_0"
MAX_NEW_TOKENS = 64

# Marcadores ChatML de Qwen
INSTRUCTION_PART = "<|im_start|>user\n"
RESPONSE_PART = "<|im_start|>assistant\n"
TOKEN_FIN_TURNO = "<|im_end|>"


# ------------------------------ Normalización --------------------------------
def normalizar_entrada(texto) -> str:
    """Mismo tratamiento en entrenamiento, prueba y app: sin espacios extra y en minúsculas."""
    return " ".join(str(texto).strip().lower().split())


def extraer_par(item: dict):
    """Devuelve (entrada, salida) o None si el ejemplo no sirve."""
    if not isinstance(item, dict) or "NOTA" in item:
        return None
    entrada = item.get("input") or item.get("palabras") or item.get("instruction") or ""
    salida = item.get("output") or item.get("oracion") or item.get("response") or ""
    if not (entrada and salida):
        return None
    return normalizar_entrada(entrada), str(salida).strip()


# ------------------------- Lectura de configuración --------------------------
def leer_configuracion(ruta_config: Path) -> dict:
    with ruta_config.open("r", encoding="utf-8-sig") as archivo:
        configuracion = json.load(archivo)
    if isinstance(configuracion, list):
        configuracion = next((item for item in configuracion if isinstance(item, dict)), None)
    if not isinstance(configuracion, dict):
        raise ValueError(f"Configuración no válida en {ruta_config}.")
    return configuracion


def resolver_archivo_configurado(ruta_raiz: Path, carpeta_base: Path, valor: str):
    if not isinstance(valor, str) or not valor.strip():
        return None
    ruta_relativa = Path(valor)
    if ruta_relativa.is_absolute() or ".." in ruta_relativa.parts:
        return None
    raiz = ruta_raiz.resolve()
    for carpeta in (carpeta_base, *carpeta_base.parents):
        carpeta_resuelta = carpeta.resolve()
        if carpeta_resuelta != raiz and raiz not in carpeta_resuelta.parents:
            continue
        ruta_candidata = (carpeta_resuelta / ruta_relativa).resolve()
        if raiz not in ruta_candidata.parents:
            continue
        if ruta_candidata.is_file():
            return ruta_candidata
    return None


def buscar_datasets(ruta_raiz: Path) -> list:
    if not ruta_raiz.exists():
        return []
    datasets = []
    for ruta_config in sorted(ruta_raiz.rglob("Package.json")):
        try:
            configuracion = leer_configuracion(ruta_config)
        except (OSError, json.JSONDecodeError, ValueError):
            continue
        ruta_dataset = resolver_archivo_configurado(
            ruta_raiz, ruta_config.parent, configuracion.get("Dataset")
        )
        if ruta_dataset is None:
            continue
        idioma = str(configuracion.get("Language") or ruta_config.parent.name)
        datasets.append({
            "nombre": ruta_dataset.stem,
            "etiqueta": f"{idioma} / {ruta_dataset.stem}",
            "idioma": idioma,
            "ruta": ruta_dataset,
            "ruta_dataset_configurada": configuracion["Dataset"],
            "carpeta_paquete": ruta_config.parent,
            "ruta_config": ruta_config,
            "configuracion": configuracion,
        })
    return datasets


def cargar_json(ruta: Path) -> list:
    with ruta.open("r", encoding="utf-8-sig") as f:
        contenido = json.load(f)
    datos = contenido.get("root", []) if isinstance(contenido, dict) else contenido
    if not isinstance(datos, list) or not datos:
        raise ValueError(f"El dataset {ruta.name} está vacío o presenta una estructura no soportada.")
    return datos


# ------------------------------ Gestor del modelo ----------------------------
class GestorModelo:
    def __init__(self):
        self.model = None
        self.tokenizer = None
        self.trainer = None
        self.pares = []
        self.ultimo_modelo_exportado = None
        self.ultimo_gguf_exportado = None

    def _liberar_memoria(self):
        self.trainer = None
        self.model = None
        self.tokenizer = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def entrenar(self, dataset_info: dict):
        print(f"🚀 Iniciando secuencia de carga para el modelo base {MODELO_BASE}...")
        lista_datos = cargar_json(dataset_info["ruta"])

        self.pares = [p for p in (extraer_par(i) for i in lista_datos) if p]
        omitidos = len(lista_datos) - len(self.pares)
        if not self.pares:
            raise ValueError("El JSON no contiene claves reconocibles.")

        self._liberar_memoria()

        usa_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        print(f"🖥️ GPU: {gpu} | precisión: {'bf16' if usa_bf16 else 'fp16'}")
        print("Vigila que la pérdida no sea 'nan'.")

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODELO_BASE,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=True,
        )
        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=LORA_R,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            lora_alpha=LORA_ALPHA,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=3407,
        )

        # Chat template nativo de Qwen3 (ChatML): es el que llevará el GGUF.
        # Sin get_chat_template: el tokenizador ya lo trae.
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = "<|endoftext|>"

        textos = []
        for entrada, salida in self.pares:
            conversacion = [
                {"role": "user", "content": entrada},
                {"role": "assistant", "content": salida},
            ]
            texto = self.tokenizer.apply_chat_template(
                conversacion, tokenize=False, add_generation_prompt=False
            )
            textos.append(texto)

        pasos_por_epoca = max(1, len(textos) // (BATCH_SIZE * ACUMULACION))
        print(f"📚 {len(textos)} ejemplos preparados ({omitidos} omitidos). "
              f"~{pasos_por_epoca * EPOCAS} pasos de entrenamiento en total.")
        print(f"Ejemplo formateado:\n{textos[0]}\n")
        dataset = Dataset.from_dict({"text": textos})

        config = SFTConfig(
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LENGTH,
            packing=False,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=ACUMULACION,
            warmup_steps=10,
            num_train_epochs=EPOCAS,
            learning_rate=LEARNING_RATE,
            fp16=not usa_bf16,
            bf16=usa_bf16,
            logging_steps=5,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir="outputs",
            report_to="none",
            disable_tqdm=True,
            dataset_num_proc=2,
        )

        kwargs = dict(model=self.model, train_dataset=dataset, args=config)
        try:
            trainer = SFTTrainer(processing_class=self.tokenizer, **kwargs)
        except TypeError:
            trainer = SFTTrainer(tokenizer=self.tokenizer, **kwargs)

        # Pérdida solo sobre la respuesta del modelo.
        trainer = train_on_responses_only(
            trainer,
            instruction_part=INSTRUCTION_PART,
            response_part=RESPONSE_PART,
        )
        self.trainer = trainer

        # Verificación de la máscara: solo debe verse la respuesta (lo demás en blanco).
        try:
            ids = trainer.train_dataset[0]["labels"]
            pad = self.tokenizer.pad_token_id
            visible = self.tokenizer.decode([pad if x == -100 else x for x in ids])
            print(f"🔎 Texto sobre el que se calcula la pérdida:\n{visible.strip()}\n")
        except Exception as error:
            print(f"(No se pudo verificar la máscara: {error})")

        stats = trainer.train()
        print(f"📉 Pérdida final de entrenamiento: {stats.training_loss:.4f}")

        # ------------------------- Exportación GGUF --------------------------
        nombre_salida = f"{NOMBRE_FAMILIA}-{dataset_info['idioma']}-{dataset_info['nombre']}"
        self.ultimo_modelo_exportado = nombre_salida
        self.ultimo_gguf_exportado = None

        gguf_antes = {
            ruta.resolve(): ruta.stat().st_mtime_ns
            for ruta in Path.cwd().rglob("*.gguf")
            if ruta.is_file()
        }
        print(f"📦 Exportando GGUF ({CUANTIZACION_GGUF})... (requiere internet activado)")
        self.model.save_pretrained_gguf(
            nombre_salida, self.tokenizer, quantization_method=CUANTIZACION_GGUF
        )
        gguf_nuevos = [
            ruta for ruta in Path.cwd().rglob("*.gguf")
            if ruta.is_file() and ruta.stat().st_mtime_ns > gguf_antes.get(ruta.resolve(), -1)
        ]
        if not gguf_nuevos:
            raise FileNotFoundError(
                f"Unsloth terminó la exportación, pero no se encontró un GGUF nuevo en {Path.cwd()}."
            )
        self.ultimo_gguf_exportado = max(
            gguf_nuevos, key=lambda ruta: ruta.stat().st_mtime_ns
        ).resolve()
        print(f"🎉 Proceso finalizado: {self.ultimo_gguf_exportado}")

    def probar(self, input_text: str) -> str:
        if self.model is None or self.tokenizer is None:
            raise RuntimeError("Entrena el modelo antes de procesar muestras.")
        FastLanguageModel.for_inference(self.model)

        mensajes = [{"role": "user", "content": normalizar_entrada(input_text)}]
        # Mismo template que en entrenamiento (sin bloque <think>).
        prompt = self.tokenizer.apply_chat_template(
            mensajes, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(
            prompt, add_special_tokens=False, return_tensors="pt"
        ).to("cuda")

        fin_turno = self.tokenizer.convert_tokens_to_ids(TOKEN_FIN_TURNO)
        eos_ids = [i for i in {self.tokenizer.eos_token_id, fin_turno} if i is not None]
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=True,
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        # Solo decodificamos los tokens nuevos (sin el prompt).
        nuevos = outputs[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(nuevos, skip_special_tokens=True).strip()

    def evaluar(self, mostrar_todos: bool = False):
        """Reproduce cada ejemplo del dataset con greedy y compara con la salida esperada."""
        if not self.pares:
            raise RuntimeError("Entrena el modelo antes de evaluar.")
        aciertos = 0
        for entrada, esperado in self.pares:
            obtenido = self.probar(entrada)
            ok = obtenido.strip() == esperado.strip()
            aciertos += ok
            if mostrar_todos or not ok:
                print(f"{'✅' if ok else '❌'} {entrada}\n"
                      f"   esperado: {esperado}\n"
                      f"   obtenido: {obtenido}\n")
        print(f"📊 Exactos: {aciertos}/{len(self.pares)} "
              f"({100 * aciertos / len(self.pares):.0f}%)")


# ------------------------------ Empaquetado ZIP ------------------------------
def ruta_paquete_segura(carpeta: Path, valor: str, campo: str):
    ruta_relativa = Path(valor)
    if ruta_relativa.is_absolute() or ".." in ruta_relativa.parts:
        raise ValueError(f"La ruta {campo} debe ser relativa y permanecer dentro de la carpeta del idioma.")
    ruta_completa = (carpeta / ruta_relativa).resolve()
    if carpeta.resolve() not in ruta_completa.parents:
        raise ValueError(f"La ruta {campo} sale de la carpeta del idioma.")
    return ruta_relativa, ruta_completa


def empaquetar_modelo(dataset_info: dict, gestor: GestorModelo) -> Path:
    if not gestor.ultimo_gguf_exportado or not gestor.ultimo_gguf_exportado.is_file():
        raise RuntimeError("No hay un GGUF generado por el último entrenamiento.")
    configuracion = dataset_info["configuracion"]
    nombre_lista = configuracion.get("WordList")
    nombre_dataset = configuracion.get("Dataset")
    nombre_modelo = configuracion.get("ModelFile")
    if not nombre_lista or not nombre_dataset or not nombre_modelo:
        raise ValueError("Package.json debe definir WordList, Dataset y ModelFile.")

    carpeta_paquete = dataset_info["carpeta_paquete"]
    ruta_lista_relativa, ruta_lista = ruta_paquete_segura(carpeta_paquete, nombre_lista, "WordList")
    ruta_modelo_relativa, _ = ruta_paquete_segura(carpeta_paquete, nombre_modelo, "ModelFile")
    ruta_dataset = dataset_info["ruta"].resolve()
    carpeta_dataset = ruta_dataset.parent
    if not ruta_lista.is_file():
        raise FileNotFoundError(f"No se encontró la lista de palabras: {ruta_lista}")
    if not ruta_dataset.is_file():
        raise FileNotFoundError(f"No se encontró el dataset configurado: {ruta_dataset}")

    ruta_zip_nombre = f"{dataset_info['idioma'].replace(' ', '_')}_{ruta_modelo_relativa.stem}.zip"
    ruta_zip = Path.cwd() / ruta_zip_nombre
    modelo_en_carpeta_dataset = (carpeta_dataset / ruta_modelo_relativa).resolve()
    rutas_incluidas = set()
    with zipfile.ZipFile(ruta_zip, "w", compression=zipfile.ZIP_DEFLATED) as archivo_zip:
        for archivo in sorted(carpeta_dataset.rglob("*")):
            if not archivo.is_file() or archivo.resolve() == modelo_en_carpeta_dataset:
                continue
            nombre_en_zip = archivo.relative_to(carpeta_dataset).as_posix()
            archivo_zip.write(archivo, arcname=nombre_en_zip)
            rutas_incluidas.add(archivo.resolve())

        if dataset_info["ruta_config"].resolve() not in rutas_incluidas:
            archivo_zip.write(dataset_info["ruta_config"], arcname="Package.json")
        if ruta_lista.resolve() not in rutas_incluidas:
            archivo_zip.write(ruta_lista, arcname=ruta_lista_relativa.as_posix())
        archivo_zip.write(gestor.ultimo_gguf_exportado, arcname=ruta_modelo_relativa.as_posix())
    return ruta_zip


# ------------------------------ Utilidades de entrada ------------------------
def procesar_entrada_json(raw_input: str) -> list:
    """Acepta un objeto, una lista JSON o una muestra JSON por línea."""
    raw_input = raw_input.strip()
    if not raw_input:
        return []
    try:
        datos = json.loads(raw_input)
        if isinstance(datos, dict):
            return [datos]
        if isinstance(datos, list):
            return [item for item in datos if isinstance(item, dict)]
    except json.JSONDecodeError:
        pass
    muestras = []
    for linea in raw_input.splitlines():
        linea = linea.strip()
        if not linea:
            continue
        try:
            muestra = json.loads(linea)
            if isinstance(muestra, dict):
                muestras.append(muestra)
        except json.JSONDecodeError:
            continue
    return muestras or [{"palabras": raw_input}]


# ----------------------------------- Interfaz --------------------------------
def construir_interfaz():
    gestor = GestorModelo()
    datasets_disponibles = buscar_datasets(RUTA_BASE)
    historial_evaluacion = []
    header = widgets.HTML(f"<h2>🎛️ {NOMBRE_FAMILIA}: entrenamiento y prueba JSON</h2>")
    if not datasets_disponibles:
        display(header, HTML("<b>⚠️ No se encontraron Package.json con Dataset válido en /kaggle/input/.</b>"))
        return
    opciones = {d["etiqueta"]: d for d in datasets_disponibles}
    dropdown = widgets.Dropdown(options=list(opciones.keys()), description="Dataset:")
    btn_entrenar = widgets.Button(description="1. Entrenar", button_style="primary")
    out_entrenar = widgets.Output()
    btn_evaluar = widgets.Button(description="1b. Evaluar en dataset", button_style="info", disabled=True)
    out_evaluar = widgets.Output()
    entrada = widgets.Textarea(
        description="Muestras:",
        placeholder='Una muestra por línea: {"palabras":"hambre pasta"}\nTambién acepta una lista JSON.',
        disabled=True, layout=widgets.Layout(width="95%", height="130px"),
    )
    btn_probar = widgets.Button(description="2. Procesar muestras", button_style="success", disabled=True)
    out_probar = widgets.Output()
    btn_descargar = widgets.Button(description="3. Crear ZIP", button_style="warning", disabled=True)
    out_descargar = widgets.Output()

    def al_entrenar(_):
        btn_entrenar.disabled = True
        with out_entrenar:
            out_entrenar.clear_output()
            try:
                gestor.entrenar(opciones[dropdown.value])
                print("✅ Entrenamiento terminado. Pulsa 'Evaluar en dataset' o introduce muestras.")
                btn_probar.disabled = False
                btn_descargar.disabled = False
                btn_evaluar.disabled = False
                entrada.disabled = False
            except Exception as error:
                print(f"❌ Error: {error}")
            finally:
                btn_entrenar.disabled = False

    def al_evaluar(_):
        btn_evaluar.disabled = True
        with out_evaluar:
            out_evaluar.clear_output()
            try:
                gestor.evaluar()
            except Exception as error:
                print(f"❌ Error al evaluar: {error}")
            finally:
                btn_evaluar.disabled = False

    def al_probar(_):
        btn_probar.disabled = True
        with out_probar:
            out_probar.clear_output()
            texto = entrada.value.strip()
            if texto.lower() == "salir":
                print(json.dumps(historial_evaluacion, ensure_ascii=False, indent=2))
                entrada.value = ""
                btn_probar.disabled = False
                return
            muestras = procesar_entrada_json(texto)
            resultados = []
            for muestra in muestras:
                palabras = muestra.get("palabras") or muestra.get("input") or muestra.get("instruction")
                if not palabras:
                    continue
                try:
                    oracion = gestor.probar(str(palabras))
                    resultado = {"palabras": str(palabras), "oracion": oracion}
                    resultados.append(resultado)
                    historial_evaluacion.append(resultado)
                except Exception as error:
                    print(f"❌ Error procesando '{palabras}': {error}")
            print("\n".join(json.dumps(r, ensure_ascii=False) for r in resultados))
            entrada.value = ""
            btn_probar.disabled = False

    def al_descargar(_):
        btn_descargar.disabled = True
        with out_descargar:
            out_descargar.clear_output()
            try:
                ruta_zip = empaquetar_modelo(opciones[dropdown.value], gestor)
                if not ruta_zip.is_file():
                    raise FileNotFoundError(f"No se encontró el ZIP generado: {ruta_zip}")
                print(f"📦 Paquete creado: {ruta_zip}")
                print("ℹ️ En Kaggle, descárgalo desde el panel Output (/kaggle/working).")
                display(FileLink(ruta_zip.name, result_html_prefix="⬇️ Descargar ZIP: "))
            except Exception as error:
                print(f"❌ Error al crear el ZIP: {error}")
            finally:
                btn_descargar.disabled = False

    btn_entrenar.on_click(al_entrenar)
    btn_evaluar.on_click(al_evaluar)
    btn_probar.on_click(al_probar)
    btn_descargar.on_click(al_descargar)
    display(widgets.VBox([
        header, dropdown, btn_entrenar, out_entrenar,
        btn_evaluar, out_evaluar,
        entrada, btn_probar, out_probar, btn_descargar, out_descargar,
    ]))


construir_interfaz()